In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [53]:
import warnings
warnings.filterwarnings('ignore')

In [54]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
        tf_device = "/GPU:0"
        print("Using TensorFlow device:", tf_device)
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)
else:
    tf_device = "/CPU:0"
    print("No GPU devices found. Using TensorFlow device:", tf_device)

1 Physical GPUs, 1 Logical GPUs
Using TensorFlow device: /GPU:0


In [55]:
df = pd.read_csv('/content/qoute_dataset.csv')

In [56]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [57]:
df.shape

(3038, 2)

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   3038 non-null   object
 1   Author  3038 non-null   object
dtypes: object(2)
memory usage: 47.6+ KB


In [59]:
df.describe()

,quote,Author
count,3038,3038
unique,3037,1005
top,A woman's heart should be so hidden in God tha...,"Cassandra Clare,"
freq,2,92


In [60]:
df.isnull().sum().sum()

np.int64(0)

In [61]:
quotes = df['quote']

In [62]:
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [63]:
quotes = quotes.str.lower()

In [64]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,"“it is our choices, harry, that show what we t..."
2,“there are only two ways to live your life. on...
3,"“the person, be it gentleman or lady, who has ..."
4,"“imperfection is beauty, madness is genius and..."


In [65]:
# remove puctuations with an empty space

import string

translator = str.maketrans('', '', string.punctuation)

quotes = quotes.apply(lambda x: x.translate(translator))

In [66]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [67]:
# Tokenization

from tensorflow.keras.preprocessing.text import Tokenizer

In [68]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [69]:
word_index = tokenizer.word_index
len(word_index)


8978

In [70]:
list(word_index.items())[:10]

[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [71]:
sequence = tokenizer.texts_to_sequences(quotes)

In [72]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [73]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [74]:
X = []
y = []

for seq in sequence:
  for i in range(1, len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]

    X.append(input_seq)
    y.append(output_seq)


In [75]:
X

[[713],
 [713, 62],
 [713, 62, 29],
 [713, 62, 29, 19],
 [713, 62, 29, 19, 16],
 [713, 62, 29, 19, 16, 946],
 [713, 62, 29, 19, 16, 946, 10],
 [713, 62, 29, 19, 16, 946, 10, 7],
 [713, 62, 29, 19, 16, 946, 10, 7, 5],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  104,
  752],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  

In [76]:
len(X)

85271

In [77]:
len(y)

85271

In [78]:
max_length = max(len(x) for x in X)
max_length

745

In [79]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_padded = pad_sequences(sequence, maxlen=max_length, padding="pre")

In [80]:
X_padded

array([[   0,    0,    0, ...,  752,   70, 2461],
       [   0,    0,    0, ...,   54,   70, 3676],
       [   0,    0,    0, ...,    7,    5, 3677],
       ...,
       [   0,    0,    0, ...,   26, 2411, 8978],
       [   0,    0,    0, ...,   17,    1,  174],
       [   0,    0,    0, ...,    3,  169,  101]], dtype=int32)

In [81]:
y = np.array(y)

In [82]:
y.shape

(85271,)

In [83]:
X_padded.shape

(3038, 745)

In [84]:
# Apply one hot encoding to y

from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [85]:
y_one_hot

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [86]:
y_one_hot.shape

(85271, 10000)

## Create Models

### RNN

In [87]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [88]:
embedding_dim = 50

In [89]:
# Total units you want to keep in hidden layers
rnn_units=128

In [90]:
rnn_model = Sequential()

rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))

rnn_model.add(SimpleRNN(units=rnn_units)) # hidden layers
rnn_model.add(Dense(units=vocab_size, activation='softmax')) # Output layers

In [91]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [92]:
rnn_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Create LSTM Model

In [109]:
lstm_units=128

In [110]:
lstm_model = Sequential()

lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))

lstm_model.add(SimpleRNN(units=lstm_units)) # hidden layers
lstm_model.add(Dense(units=vocab_size, activation='softmax')) # Output layers

In [111]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [112]:
lstm_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_7 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Train the model

### RNN

In [96]:
epochs = 100
batch_size = 128

In [97]:
history_rnn = rnn_model.fit(X_padded, y_one_hot, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=1)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 179ms/step - accuracy: 0.0154 - loss: 8.5062 - val_accuracy: 0.0230 - val_loss: 7.0663
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 0.0278 - loss: 6.1712 - val_accuracy: 0.0822 - val_loss: 6.5346
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.0384 - loss: 5.8555 - val_accuracy: 0.0822 - val_loss: 6.6192
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.0384 - loss: 5.8196 - val_accuracy: 0.0822 - val_loss: 6.6692
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.0315 - loss: 5.8047 - val_accuracy: 0.0822 - val_loss: 6.7437
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.0384 - loss: 5.8001 - val_accuracy: 0.0822 - val_loss: 6.7766
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.0300 - loss: 5.7979 - val_accuracy: 0.0822 - val_loss: 6.8141
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.0384 - loss: 5.7924 - val_accuracy: 0

###LSTM

In [113]:
epochs = 100
batch_size = 128

In [114]:
history_lstm = lstm_model.fit(X_padded, y_one_hot, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=1)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 182ms/step - accuracy: 0.0219 - loss: 8.2991 - val_accuracy: 0.0099 - val_loss: 6.8900
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.0304 - loss: 6.0982 - val_accuracy: 0.0822 - val_loss: 6.5465
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.0384 - loss: 5.8519 - val_accuracy: 0.0822 - val_loss: 6.6362
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.0304 - loss: 5.8162 - val_accuracy: 0.0822 - val_loss: 6.6877
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.0384 - loss: 5.8025 - val_accuracy: 0.0822 - val_loss: 6.7519
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.0384 - loss: 5.7974 - val_accuracy: 0.0822 - val_loss: 6.7630
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - accuracy: 0.0384 - loss: 5.7927 - val_accuracy: 0.0822 - val_loss: 6.8192
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.0384 - loss: 5.7927 - val_accuracy: 0

In [116]:
# Save Models

lstm_model.save("lstm_model.h5")
rnn_model.save("rnn_model.h5")

## Prediction

In [118]:
index_to_word = {}

for word, index in word_index.items():
  index_to_word[index] = word

In [119]:
index_to_word

{1: 'the',
 2: 'you',
 3: 'to',
 4: 'and',
 5: 'a',
 6: 'i',
 7: 'is',
 8: 'of',
 9: 'that',
 10: 'it',
 11: 'in',
 12: 'be',
 13: 'not',
 14: 'are',
 15: 'your',
 16: 'have',
 17: 'for',
 18: 'but',
 19: 'we',
 20: 'if',
 21: 'what',
 22: 'with',
 23: 'all',
 24: 'love',
 25: 'can',
 26: 'my',
 27: 'when',
 28: 'will',
 29: 'as',
 30: 'who',
 31: 'do',
 32: 'or',
 33: 'me',
 34: 'he',
 35: 'they',
 36: 'life',
 37: 'one',
 38: 'was',
 39: 'like',
 40: 'there',
 41: 'people',
 42: 'on',
 43: 'its',
 44: 'at',
 45: 'so',
 46: 'never',
 47: 'no',
 48: 'them',
 49: 'dont',
 50: 'know',
 51: 'just',
 52: 'more',
 53: 'only',
 54: 'than',
 55: 'because',
 56: 'this',
 57: 'want',
 58: 'up',
 59: 'how',
 60: 'his',
 61: 'things',
 62: 'world',
 63: 'by',
 64: 'think',
 65: 'make',
 66: 'about',
 67: 'time',
 68: 'from',
 69: 'always',
 70: 'our',
 71: 'an',
 72: 'out',
 73: 'us',
 74: 'good',
 75: 'said',
 76: 'she',
 77: 'her',
 78: 'way',
 79: 'go',
 80: 'am',
 81: 'live',
 82: 'has',
 83:

In [120]:
def predictor(model, tokenizer, text, max_length):

  text = text.lower()
  seq = tokenizer.texts_to_sequences([text])[0]
  padded_seq = pad_sequences([seq], maxlen=max_length, padding="pre")

  pred = model.predict(padded_seq, verbose=0)
  pred_index = np.argmax(pred)
  predicted_word = index_to_word[pred_index]

  return predicted_word



In [124]:
seed_text = 'what are you trying'

predicted_word = predictor(lstm_model, tokenizer, seed_text, max_length)
print(predicted_word)

read


In [123]:
seed_text = 'life is not only'

predicted_word = predictor(lstm_model, tokenizer, seed_text, max_length)
print(predicted_word)

learn


In [138]:
# Predict a full sentence

def generate_text(model, tokenizer, seed_text, max_length, n_words):

  for _ in range(n_words):
    predicted_word = predictor(model, tokenizer, seed_text, max_length)
    if predicted_word == "":
      break
    seed_text += " " + predicted_word
  return seed_text


In [140]:
seed_text = "The person is"

generated_text = generate_text(lstm_model, tokenizer, seed_text, max_length, 10)
print(generated_text)

The person is little as but that but at may a with of


In [141]:
seed_text = "The measured quantity"

generated_text = generate_text(lstm_model, tokenizer, seed_text, max_length, 10)
print(generated_text)

The measured quantity is little i so it that if with day is


In [143]:
import pickle

# wb -> write binary

with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)


with open("max_length.pkl", "wb") as f:
  pickle.dump(max_length, f)